# Capstone — Predicting Search Traffic Decay: A Decision-Support Model for Content Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muzammilsharf/flyrank-ml-internship/blob/main/work/notebooks/capstone.ipynb)

This notebook is the reproducible source behind the deployed paper (see `submission/paper_url.txt`). It mirrors the paper's structure section by section, loading the same saved dataset built in w05/w06/w07 rather than re-running the full warehouse pull.


## 1. Introduction

*See the deployed paper for the full framing. Lane: Refresh / Content Opportunity Scoring. Decision supported: a weekly editorial review queue, ranking pages by likelihood of forward search-traffic decline.*

In [1]:
import pandas as pd
import json
import numpy as np
import lightgbm as lgb
from sklearn.model_selection import GroupShuffleSplit
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

# Load the dataset built and validated across w05/w06 -- no need to re-pull the warehouse.
model_df_encoded = pd.read_parquet('data/flyrank_snapshots.parquet')
with open('data/flyrank_meta.json') as f:
    features = json.load(f)["features"]

print(model_df_encoded.shape)


(357671, 47)


## 2. Data

See the deployed paper, Section 2, for the full data description, exclusions, and date-window rationale. Confirming scope here:

In [2]:
print(f"Total eligible page-month snapshots: {len(model_df_encoded)}")
print(f"Unique clients: {model_df_encoded['client_hash_id'].nunique()}")
print(f"Date range: {model_df_encoded['report_date'].min()} to {model_df_encoded['report_date'].max()}")
print(f"Overall label rate: {model_df_encoded['is_declining_label'].mean():.3f}")


Total eligible page-month snapshots: 357671
Unique clients: 36
Date range: 2025-05-14 00:00:00 to 2026-04-30 00:00:00
Overall label rate: 0.362


## 3. Methodology

Label: forward-looking 30-day decline vs. the trailing 30 days, minimum 100 impressions. Baseline: transparent CTR-vs-position heuristic (0.240 Precision@50). Validation: client-level holdout, averaged across 5 seeds. Full reasoning and the two rejected split designs are documented in `w06_validation_audit.ipynb`.

## 4. Results

Four-model comparison, single split, then the corrected multi-seed evaluation.

In [3]:
def precision_at_50(y_true, scores):
    top50_idx = pd.Series(scores).nlargest(50).index
    return y_true.iloc[top50_idx].mean()

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))
train, test = model_df_encoded.iloc[train_idx], model_df_encoded.iloc[test_idx]

X_train, y_train = train[features], train["is_declining_label"]
X_test, y_test = test[features], test["is_declining_label"]
X_train_filled, X_test_filled = X_train.fillna(0), X_test.fillna(0)

results = {}
lr = LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42).fit(X_train_filled, y_train)
results["Logistic Regression"] = precision_at_50(y_test.reset_index(drop=True), lr.predict_proba(X_test_filled)[:, 1])

dt = DecisionTreeClassifier(max_depth=8, class_weight="balanced", random_state=42).fit(X_train_filled, y_train)
results["Decision Tree"] = precision_at_50(y_test.reset_index(drop=True), dt.predict_proba(X_test_filled)[:, 1])

rf = RandomForestClassifier(n_estimators=200, max_depth=10, class_weight="balanced", random_state=42, n_jobs=-1).fit(X_train_filled, y_train)
results["Random Forest"] = precision_at_50(y_test.reset_index(drop=True), rf.predict_proba(X_test_filled)[:, 1])

gbm = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31, learning_rate=0.05,
                           is_unbalance=True, random_state=42, verbosity=-1).fit(X_train, y_train)
results["LightGBM"] = precision_at_50(y_test.reset_index(drop=True), gbm.predict_proba(X_test)[:, 1])

print(pd.DataFrame({"Model": results.keys(), "Precision@50": [f'{v:.3f}' for v in results.values()]}).to_string(index=False))
print("Baseline (heuristic): 0.240")


/home/muzammil/flyrank-ml-internship/venv/lib64/python3.14/site-packages/sklearn/linear_model/_logistic.py:599: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


              Model Precision@50
Logistic Regression        0.540
      Decision Tree        0.920
      Random Forest        0.260
           LightGBM        0.960
Baseline (heuristic): 0.240


In [4]:
# The corrected, reported result: mean across 5 client-holdout splits
seeds = [42, 7, 123, 2024, 99]
scores = []
spike_months = ["2026-03", "2026-04"]

for seed in seeds:
    gss_s = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=seed)
    tr_idx, te_idx = next(gss_s.split(model_df_encoded, groups=model_df_encoded["client_hash_id"]))
    tr, te = model_df_encoded.iloc[tr_idx], model_df_encoded.iloc[te_idx]
    te = te[~te["report_date"].dt.to_period("M").astype(str).isin(spike_months)]
    if len(te) < 50:
        continue
    gbm_s = lgb.LGBMClassifier(n_estimators=300, max_depth=6, num_leaves=31, learning_rate=0.05,
                                 is_unbalance=True, random_state=42, verbosity=-1).fit(tr[features], tr["is_declining_label"])
    te = te.copy()
    te["risk"] = gbm_s.predict_proba(te[features])[:, 1]
    scores.append(te.nlargest(50, "risk")["is_declining_label"].mean())

print(f"Per-seed: {[f'{s:.3f}' for s in scores]}")
print(f"Mean: {np.mean(scores):.3f}, Std: {np.std(scores):.3f}, Range: {min(scores):.3f}-{max(scores):.3f}")


Per-seed: ['0.880', '0.820', '0.560', '0.900', '0.780']
Mean: 0.788, Std: 0.122, Range: 0.560-0.900


## 5. Limitations & honest framing

See the deployed paper, Section 5, for the full list (not causal, anomalous-period caveat, small-sample variance, uneven analytics coverage, excluded client). Restated briefly: this model ranks pages for human review; it does not predict individual outcomes with certainty, and its performance during the excluded March-April 2026 spike period is untested.

## 6. Ranked recommendations

See `w07_action_playbook.ipynb` for the full reason-code logic and export pipeline. Summary: pages are ranked by risk score and tagged with one of four reason codes (`SEVERE_CTR_DEFICIT`, `PAGE_ONE_OPPORTUNITY`, `LOW_ENGAGEMENT`, `GENERAL_DECLINE_RISK`), each mapped to a suggested review action. Human review and the no-go list (never auto-publish, never auto-delete, never treat the score as a public claim) apply to every recommendation.

## 7. Reproducibility

- `w04_baseline_score.ipynb` -- heuristic baseline
- `w05_model.ipynb` -- label design, feature pipeline, model comparison
- `w06_validation_audit.ipynb` -- split correction, leakage audit
- `w07_action_playbook.ipynb` -- ranked queue, reason codes, exports
- This notebook -- full narrative and reproducible results

Fixed seeds throughout: model training `random_state=42`; validation sweep `[42, 7, 123, 2024, 99]`.


## 5-Minute Demo Outline (Week 8 Showcase)

**Question (30 sec):** Content teams can't review every page every week. Can we rank pages by risk of losing search traffic, so editors know which ones to check first?

**Method (1 min):** 18 months of real FlyRank search data. LightGBM trained on 90 days of traffic and content signals per page, predicting a 30-day forward decline. Tested on entire clients the model had never seen, not just individual pages, to keep the result honest.

**One chart (1 min):** *[show the model-comparison bar chart from the paper's Results section]* — baseline heuristic at 24%, final model at 78.8% average.

**One honest result (1 min):** 78.8% mean Precision@50, but with real variance (56%–94%) across five different test splits, because only 36 clients exist in this data. Also: an early number looked too good (94%) and turned out to be the model detecting a real traffic-drop event instead of learning general page risk, worth telling this story since it shows the debugging, not just the win.

**One recommendation (1 min):** editors get a ranked weekly list with reason codes (why each page was flagged), not a black-box score, and clear rules for what should never be automated (auto-publishing, auto-deleting).

## Shareable Cuts

**Social post (methodology-focused):**

Spent the last few weeks building a model to predict search traffic decline for content pages, using real production data. The hardest part wasn't the model, it was catching my own mistakes: a filter that silently dropped most of my data, a feature that leaked future information, and a "great" 94% score that turned out to be the model cheating by spotting a specific month instead of learning real patterns. Fixed all three, ended with an honest 78.8% (baseline: 24%), reported as a range, not a single flattering number. Full writeup + reproducible code: [your paper URL]

**Employer-facing summary (3 sentences):**

I built a machine learning model that predicts which web pages are losing search traffic, using 18 months of real search and analytics data across 36 clients. After finding and fixing three data-leakage issues during validation, the model correctly identified declining pages 78.8% of the time in its top recommendations, more than 3x better than a simple rule-based baseline. The project includes a full reproducible pipeline, an honest reporting of the model's limits, and a ranked action queue designed for human review, not automated decisions.

## Acknowledgments

Built on the FlyRank ML Internship dataset -- [flyrank.ai](https://flyrank.ai).